# 22. DB-API: Connections & Cursors

*Module: 6 — Python access · Companion summary: `lectures/22-database-api-connections.md` · SQL engine: SQLite (in-memory)*

## What you will study

- walk the five DB-API steps
- fetch results (fetchall / fetchone), parameterize queries (? placeholders)
- commit vs rollback — and why closing matters
- use a context manager so cleanup is automatic

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

## 🧪 Five steps against a *real file* database

Using `tempfile` so the notebook is re-runnable without leaving junk behind:

In [ ]:
import os, sqlite3, tempfile

db_dir = tempfile.mkdtemp()                            # fresh folder per run
db_path = os.path.join(db_dir, 'demo.db')
print('database file:', db_path)

conn = sqlite3.connect(db_path)                        # STEP 1-2: open
conn.execute("CREATE TABLE person (id INTEGER PRIMARY KEY, name TEXT)")   # STEP 3/4
conn.commit()                                          # STEP 4b: persist DDL/DML
conn.execute("INSERT INTO person VALUES (1, 'Raul'), (2, 'Rav')")
conn.commit()
print('on disk now:', os.listdir(db_dir))

## 🧪 STEP 3 — cursor + fetchall / fetchone

In [ ]:
cur = conn.cursor()
print('fetchall:', cur.execute("SELECT * FROM person").fetchall())
print('fetchone:', cur.execute("SELECT * FROM person").fetchone())   # first row only
print('rowcount:', cur.execute("SELECT COUNT(*) FROM person").fetchone()[0])

## 🧪 Parameterized queries — the safe way to inject values

In [ ]:
who = 'Rav'
print(cur.execute("SELECT * FROM person WHERE name = ?", (who,)).fetchall())

🧪 executemany — bulk insert

In [ ]:
cur.executemany("INSERT INTO person VALUES (?, ?)", [(3, 'Ana'), (4, 'Jack')])
conn.commit()
print(cur.execute("SELECT * FROM person").fetchall())

## 🧪 Commit vs Rollback — with the real file on disk

In [ ]:
try:
    conn.execute("INSERT INTO person VALUES (5, 'Ghost')")
    conn.rollback()                     # pretend we changed our mind
finally:
    pass
print('after rollback:', cur.execute("SELECT * FROM person").fetchall())
conn.execute("INSERT INTO person VALUES (5, 'Real')")
conn.commit()
print('after commit:  ', cur.execute("SELECT * FROM person").fetchall())

## 🧪 STEP 5 — close, and better: the context manager

In [ ]:
conn.close()
print('cursor after close:', end=' ')
try:
    cur.execute("SELECT 1")
except sqlite3.ProgrammingError as e:
    print('refused,', e)

with sqlite3.connect(db_path) as c:      # commits on success, rolls back on error
    print('inside:', c.execute("SELECT COUNT(*) FROM person").fetchone()[0], 'persons')
    c.execute("INSERT INTO person VALUES (6, 'FromWith')")   # auto-committed on exit
c.close()                                # ⚠️ 'with' handles the TRANSACTION, not the connection!
probe = sqlite3.connect(db_path)         # fresh connection to verify persistence
print('file still on disk?', os.path.exists(db_path))
print('after with:', probe.execute("SELECT * FROM person WHERE id = 6").fetchall())
probe.close()                            # close it, or Windows blocks the cleanup below
os.remove(db_path); os.rmdir(db_dir)     # tidy up the temp folder

## 💡 Tips & tricks

- **`?` placeholders, never f-strings** — string-building SQL is the SQL-injection bug.
- `fetchall()` for small results, `fetchone()`/iterate for big ones.
- `with connection` is the shortcut for commit/rollback — but does **not** close the
  connection; pair with try/finally or `closing()` for that.

## 🎯 Your turn

1. Create table `book (id, title)` in a new temp file, insert 3 books via executemany,
   fetchone + fetchall, then tear down the file.
2. Insert inside `with` but trigger an exception — verify the row did NOT land.

In [ ]:
# 💡 Solution
import os, sqlite3, tempfile
d = tempfile.mkdtemp(); p = os.path.join(d, 'books.db')
c = sqlite3.connect(p)
with c:                                  # transaction scope: commits when the block exits
    c.execute("CREATE TABLE book (id INTEGER PRIMARY KEY, title TEXT)")
    c.executemany("INSERT INTO book VALUES (?, ?)", [(1, 'A'), (2, 'B'), (3, 'C')])
c.close()                                # 'with' did NOT close it — we do, explicitly
cur = sqlite3.connect(p)
print('fetchone:', cur.execute('SELECT * FROM book').fetchone())
print('fetchall:', cur.execute('SELECT * FROM book').fetchall())
cur.close()
os.remove(p); os.rmdir(d)                # only safe once every connection is closed

d = tempfile.mkdtemp(); p = os.path.join(d, 'rb.db')
c = sqlite3.connect(p)
try:
    with c:                                  # anything raised inside → automatic ROLLBACK
        c.execute("CREATE TABLE t (x INTEGER CHECK (x < 10))")
        c.execute("INSERT INTO t VALUES (1)")            # would be lost...
        c.execute("INSERT INTO t VALUES (99)")           # CHECK constraint fires →
        print('never reached')                           # ...because we never commit
except sqlite3.IntegrityError as e:
    print('constraint fired:', e)
c.close()                                    # with = transaction scope, NOT a close
chk = sqlite3.connect(p)
print('rows after failed txn (row 1 must be GONE):', chk.execute('SELECT * FROM t').fetchall())
chk.close()
os.remove(p); os.rmdir(d)